# Senior Video Editor — Colab
Run Cell 1 → Cell 2 → Cell 3.


In [ ]:
# ============================================================
# SENIOR VIDEO EDITOR - FINAL COLAB SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q faster-whisper rapidfuzz ipywidgets

import os
import re
import json
import shutil
import subprocess
from datetime import datetime

BASE = "/content/drive/MyDrive/SeniorVideoEditor"
PROJECTS = os.path.join(BASE, "projects")
EXPORTS = os.path.join(BASE, "exports")
TEMP = "/content/senior_video_editor_temp"

for folder in [BASE, PROJECTS, EXPORTS, TEMP]:
    os.makedirs(folder, exist_ok=True)

print("✅ Google Drive connected")
print("✅ FFmpeg ready")
print("✅ Faster-Whisper ready")
print("✅ Senior Video Editor folders ready")
print("📁 Projects:", PROJECTS)
print("🎬 Exports:", EXPORTS)


In [ ]:
# ============================================================
# SENIOR VIDEO EDITOR - PROJECT + SENTENCE TIMING ENGINE
# ============================================================

from faster_whisper import WhisperModel
from rapidfuzz.fuzz import ratio

whisper_model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

def natural_key(path):
    name = os.path.basename(path)
    return [int(x) if x.isdigit() else x.lower()
            for x in re.split(r'(\d+)', name)]

def normalize_word(word):
    return re.sub(r"[^a-zA-Z0-9']", "", str(word).lower())

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text.strip())
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def get_audio_duration(path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())

def transcribe_words(audio_path):
    segments, info = whisper_model.transcribe(
        audio_path,
        word_timestamps=True,
        vad_filter=True,
        beam_size=1
    )

    words = []
    for segment in segments:
        if not segment.words:
            continue
        for word in segment.words:
            clean = normalize_word(word.word)
            if clean:
                words.append({
                    "word": clean,
                    "start": float(word.start),
                    "end": float(word.end)
                })
    return words

def word_matches(a, b):
    if a == b:
        return True
    if len(a) <= 2 or len(b) <= 2:
        return False
    return ratio(a, b) >= 82

def build_sentence_timings(audio_path, script_text):
    sentences = split_sentences(script_text)
    words = transcribe_words(audio_path)
    audio_duration = get_audio_duration(audio_path)

    if not sentences:
        raise Exception("Script me sentences nahi mile.")
    if not words:
        raise Exception("Audio transcription nahi ban saki.")

    timings = []
    current_word = 0

    for scene_number, sentence in enumerate(sentences, start=1):
        targets = [normalize_word(w) for w in re.findall(r"[A-Za-z0-9']+", sentence)]
        targets = [w for w in targets if w]
        if not targets:
            continue

        first_targets = targets[:min(5, len(targets))]
        last_targets = targets[-min(5, len(targets)):]

        best_start = None
        best_start_score = -1
        search_limit = min(len(words), current_word + max(80, len(targets) * 4))

        for candidate in range(current_word, search_limit):
            score = 0
            for offset, target in enumerate(first_targets):
                wi = candidate + offset
                if wi >= len(words):
                    break
                if word_matches(words[wi]["word"], target):
                    score += 1
            if score > best_start_score:
                best_start_score = score
                best_start = candidate

        if best_start is None:
            best_start = current_word

        expected_end = min(len(words) - 1, best_start + len(targets) + 15)
        best_end = expected_end
        best_end_score = -1

        end_search_start = max(best_start, expected_end - 25)
        end_search_end = min(len(words), expected_end + 30)

        for candidate in range(end_search_start, end_search_end):
            score = 0
            start_compare = max(0, candidate - len(last_targets) + 1)
            actual_words = [w["word"] for w in words[start_compare:candidate + 1]]

            for target in last_targets:
                if any(word_matches(a, target) for a in actual_words):
                    score += 1

            if score > best_end_score:
                best_end_score = score
                best_end = candidate

        start_time = words[best_start]["start"]
        end_time = words[best_end]["end"]

        if timings:
            start_time = max(start_time, timings[-1]["end"])

        end_time = max(end_time, start_time + 0.25)

        timings.append({
            "scene": scene_number,
            "sentence": sentence,
            "start": round(start_time, 3),
            "end": round(end_time, 3),
            "duration": round(end_time - start_time, 3)
        })

        current_word = min(len(words) - 1, best_end + 1)

    if timings:
        timings[0]["start"] = 0.0

        for i in range(len(timings) - 1):
            next_start = timings[i + 1]["start"]
            if next_start > timings[i]["start"]:
                timings[i]["end"] = next_start
                timings[i]["duration"] = round(
                    timings[i]["end"] - timings[i]["start"], 3
                )

        timings[-1]["end"] = round(audio_duration, 3)
        timings[-1]["duration"] = round(
            max(0.25, audio_duration - timings[-1]["start"]), 3
        )

    return timings

def save_project(audio_file, image_files, script_text):
    if not audio_file:
        raise Exception("Narration audio upload karo.")
    if not image_files:
        raise Exception("Sentence images upload karo.")
    if not script_text.strip():
        raise Exception("Script paste karo.")

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    project_dir = os.path.join(PROJECTS, f"project_{stamp}")
    audio_dir = os.path.join(project_dir, "audio")
    image_dir = os.path.join(project_dir, "images")
    os.makedirs(audio_dir, exist_ok=True)
    os.makedirs(image_dir, exist_ok=True)

    audio_ext = os.path.splitext(audio_file)[1] or ".mp3"
    audio_path = os.path.join(audio_dir, "narration" + audio_ext)
    shutil.copy(audio_file, audio_path)

    saved_images = []
    for i, img in enumerate(image_files, start=1):
        ext = os.path.splitext(img)[1] or ".jpg"
        destination = os.path.join(image_dir, f"{i:04d}{ext.lower()}")
        shutil.copy(img, destination)
        saved_images.append(destination)

    script_path = os.path.join(project_dir, "script.txt")
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(script_text)

    timings = build_sentence_timings(audio_path, script_text)

    timing_path = os.path.join(project_dir, "sentence_timings.json")
    with open(timing_path, "w", encoding="utf-8") as f:
        json.dump(timings, f, indent=2, ensure_ascii=False)

    return project_dir, timings, len(saved_images)

print("✅ Project + sentence timing engine loaded")


In [ ]:
# ============================================================
# SENIOR VIDEO EDITOR - DIRECT COLAB UI + FINAL FFMPEG EXPORT
# NO GRADIO / NO PUBLIC LINK / NO LOCAL SERVER
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, Video

def render_final_video(project_dir):
    if not project_dir or not os.path.isdir(project_dir):
        raise Exception("Project folder nahi mila.")

    audio_dir = os.path.join(project_dir, "audio")
    image_dir = os.path.join(project_dir, "images")
    timings_path = os.path.join(project_dir, "sentence_timings.json")

    audio_files = [
        os.path.join(audio_dir, f)
        for f in os.listdir(audio_dir)
        if f.lower().endswith((".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg"))
    ]
    if not audio_files:
        raise Exception("Narration audio nahi mili.")

    audio_path = audio_files[0]

    images = [
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]
    images = sorted(images, key=natural_key)

    if not images:
        raise Exception("Images nahi mili.")
    if not os.path.exists(timings_path):
        raise Exception("Sentence timings file nahi mili.")

    with open(timings_path, "r", encoding="utf-8") as f:
        timings = json.load(f)

    if not timings:
        raise Exception("Sentence timings empty hain.")

    audio_duration = get_audio_duration(audio_path)
    usable = min(len(images), len(timings))

    if usable == 0:
        raise Exception("Export ke liye scenes nahi hain.")

    concat_file = os.path.join(TEMP, "final_timeline.txt")

    with open(concat_file, "w", encoding="utf-8") as f:
        for i in range(usable):
            image_path = images[i]
            duration = max(0.15, float(timings[i]["duration"]))
            safe = image_path.replace("'", "'\\''")
            f.write(f"file '{safe}'\n")
            f.write(f"duration {duration:.6f}\n")

        last_image = images[usable - 1].replace("'", "'\\''")
        f.write(f"file '{last_image}'\n")

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(EXPORTS, f"SeniorVideo_FINAL_{stamp}.mp4")

    video_filter = (
        "scale=1920:1080:force_original_aspect_ratio=increase,"
        "crop=1920:1080,"
        "fps=30,"
        "format=yuv420p"
    )

    cmd = [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", concat_file,
        "-i", audio_path,
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-vf", video_filter,
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "20",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-b:a", "192k",
        "-ar", "48000",
        "-ac", "2",
        "-t", f"{audio_duration:.6f}",
        "-movflags", "+faststart",
        output_path
    ]

    process = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if process.returncode != 0:
        raise Exception("FFmpeg failed:\n" + process.stderr[-4000:])

    final_duration = get_audio_duration(output_path)

    return {
        "output_path": output_path,
        "audio_duration": audio_duration,
        "video_duration": final_duration,
        "difference": abs(audio_duration - final_duration),
        "images_used": usable,
        "sentences": len(timings)
    }

def uploaded_items(uploader):
    value = uploader.value
    if not value:
        return []
    if isinstance(value, (tuple, list)):
        return list(value)
    if isinstance(value, dict):
        items = []
        for name, data in value.items():
            if isinstance(data, dict):
                item = dict(data)
                item.setdefault("name", name)
                items.append(item)
        return items
    return []

def save_uploaded_item(item, path):
    content = item.get("content")
    if hasattr(content, "tobytes"):
        content = content.tobytes()
    with open(path, "wb") as f:
        f.write(content)

title = widgets.HTML(
    '''
    <h2>🎬 Senior Video Editor</h2>
    <b>Exact Sentence → Image → Narration Sync</b><br>
    1 sentence = 1 image<br>
    Final MP4 saves directly to Google Drive.
    <hr>
    '''
)

audio_upload = widgets.FileUpload(
    accept=".mp3,.wav,.m4a,.aac,.flac,.ogg",
    multiple=False,
    description="Upload Audio"
)

image_upload = widgets.FileUpload(
    accept="image/*",
    multiple=True,
    description="Upload Images"
)

script_box = widgets.Textarea(
    placeholder="Paste exact narration script here...",
    description="Script:",
    layout=widgets.Layout(width="100%", height="250px")
)

create_button = widgets.Button(
    description="1️⃣ CREATE PROJECT + AUTO SYNC",
    button_style="success",
    layout=widgets.Layout(width="330px", height="45px")
)

export_button = widgets.Button(
    description="2️⃣ EXPORT FINAL MP4",
    button_style="primary",
    disabled=True,
    layout=widgets.Layout(width="330px", height="45px")
)

project_output = widgets.Output()
export_output = widgets.Output()
current_project = {"path": None}

def create_project_clicked(btn):
    with project_output:
        clear_output()

        try:
            audio_items = uploaded_items(audio_upload)
            image_items = uploaded_items(image_upload)
            script_text = (script_box.value or "").strip()

            if not audio_items:
                print("❌ Narration audio upload karo.")
                return
            if not image_items:
                print("❌ Images upload karo.")
                return
            if not script_text:
                print("❌ Script paste karo.")
                return

            temp_upload = os.path.join(TEMP, "uploads")
            if os.path.exists(temp_upload):
                shutil.rmtree(temp_upload)
            os.makedirs(temp_upload, exist_ok=True)

            audio_name = audio_items[0].get("name", "narration.mp3")
            audio_ext = os.path.splitext(audio_name)[1] or ".mp3"
            audio_path = os.path.join(temp_upload, "narration" + audio_ext)
            save_uploaded_item(audio_items[0], audio_path)

            image_paths = []
            for index, item in enumerate(image_items, start=1):
                original_name = item.get("name", f"{index}.jpg")
                ext = os.path.splitext(original_name)[1] or ".jpg"
                img_path = os.path.join(temp_upload, f"{index:04d}{ext.lower()}")
                save_uploaded_item(item, img_path)
                image_paths.append(img_path)

            print("⏳ Audio analyze ho rahi hai. Sentence timings ban rahe hain...")

            project_dir, timings, image_count = save_project(
                audio_path,
                image_paths,
                script_text
            )

            current_project["path"] = project_dir
            export_button.disabled = False

            print("\n✅ PROJECT READY")
            print("📁", project_dir)
            print("📝 Sentences:", len(timings))
            print("🖼 Images:", image_count)

            if len(timings) != image_count:
                print("\n⚠️ Sentence count aur image count same nahi hain.")

            print("\n⏱ First 10 timings:")
            print(json.dumps(timings[:10], indent=2, ensure_ascii=False))

        except Exception as e:
            print("❌ ERROR:")
            print(str(e))

def export_clicked(btn):
    with export_output:
        clear_output()

        project_dir = current_project.get("path")
        if not project_dir:
            print("❌ Pehle project create karo.")
            return

        try:
            print("🎬 Final video render start...")
            print("⏳ Long 1080p video ko time lag sakta hai. Cell/browser band mat karo.")

            result = render_final_video(project_dir)

            print("\n✅ FINAL VIDEO COMPLETE")
            print(f"🎙 Narration: {result['audio_duration']:.3f} sec")
            print(f"🎬 Video: {result['video_duration']:.3f} sec")
            print(f"⏱ Difference: {result['difference']:.3f} sec")
            print(f"📝 Sentences: {result['sentences']}")
            print(f"🖼 Images used: {result['images_used']}")
            print("\n💾 Saved to Google Drive:")
            print(result["output_path"])

            display(Video(result["output_path"], embed=False))

        except Exception as e:
            print("❌ EXPORT ERROR:")
            print(str(e))

create_button.on_click(create_project_clicked)
export_button.on_click(export_clicked)

display(
    title,
    widgets.HTML("<b>🎙 Narration Audio</b>"),
    audio_upload,
    widgets.HTML("<br><b>🖼 Sentence Images</b>"),
    image_upload,
    widgets.HTML("<br><b>📝 Full Script</b>"),
    script_box,
    widgets.HTML("<br>"),
    create_button,
    project_output,
    widgets.HTML("<hr>"),
    export_button,
    export_output
)
